# 1 Load modules

In [1]:
import pandas as pd
import numpy as np
import glob
import os

In [2]:
os.makedirs('/nfs/team151/mt19/annotation_preprocessed', exist_ok=True)


# 2 Read in Ensembl GTF

In [3]:
gtf_path = '/nfs/users/nfs_m/mt19/cardinal_analysis/ht/datasets/Flanders/trans_eQTLS_ERs/Homo_sapiens.GRCh38.110.gtf.gz'
gtf_cols = ['seqname','source','feature','start','end','score','strand','frame','attribute']
gtf = pd.read_csv(gtf_path, sep='\t', comment='#', names=gtf_cols, dtype={'seqname': str}, low_memory=False)

genes = gtf[gtf.feature == 'gene'].copy()
genes['gene_id'] = genes.attribute.str.extract(r'gene_id "([^"]+)"')
genes['gene_name'] = genes.attribute.str.extract(r'gene_name "([^"]+)"')
symbol_to_ensembl = genes[['gene_name','gene_id']].dropna().drop_duplicates()
ens2sym = symbol_to_ensembl.set_index('gene_id')['gene_name'].to_dict()
sym2ens = symbol_to_ensembl.groupby('gene_name')['gene_id'].apply(lambda s: sorted(set(s))).to_dict()

check = symbol_to_ensembl[symbol_to_ensembl.gene_name.isin(['MYC','SPI1','TERT'])]
print(check)

        gene_name          gene_id
1066461      TERT  ENSG00000164362
1654227       MYC  ENSG00000136997
1851398      SPI1  ENSG00000066336


# 3 load STRING, recombine no-coexpression score

In [4]:
string_path = '/nfs/users/nfs_m/mt19/cardinal_analysis/ht/datasets/Flanders/trans_eQTLS_ERs/string_interactions.tsv'
string = pd.read_csv(string_path, sep='\t')
string.columns = [c.lstrip('#') for c in string.columns]
print(string.shape)

p = 0.041
channels_no_coexp = ['neighborhood_on_chromosome', 'gene_fusion', 'phylogenetic_cooccurrence',
                     'experimentally_determined_interaction', 'database_annotated', 'automated_textmining']

def recombine(df, channels, prior=0.041):
    s_running = np.zeros(len(df))
    for ch in channels:
        s = df[ch].values
        s_nop = np.where(s > 0, (s - prior) / (1 - prior), 0.0)
        s_nop = np.clip(s_nop, 0, None)
        s_running = 1 - (1 - s_running) * (1 - s_nop)
    return s_running + prior * (1 - s_running)

string['combined_score_no_coexp'] = recombine(string, channels_no_coexp, p)

(116176, 13)


### calcualte an score based soely on experimentally_determined_interaction

In [5]:
p = 0.041

def single_channel_score(df, channel, prior=0.041):
    s = df[channel].values
    s_nop = np.where(s > 0, (s - prior) / (1 - prior), 0.0)
    s_nop = np.clip(s_nop, 0, None)
    return s_nop + prior * (1 - s_nop)

string['score_experimentally_determined_interaction'] = single_channel_score(
    string, 'experimentally_determined_interaction', p
)

In [6]:
print(string['score_experimentally_determined_interaction'].describe())
print(string['experimentally_determined_interaction'].describe())

for t in [0.041, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.7, 0.9]:
    n = (string['score_experimentally_determined_interaction'] >= t).sum()
    print(f'>= {t}: {n} pairs')

# also check how many have literally zero experimental evidence
print('exactly 0:', (string['experimentally_determined_interaction'] == 0).sum(), '/', len(string))

count    116176.000000
mean          0.082055
std           0.110621
min           0.041000
25%           0.041000
50%           0.041000
75%           0.074000
max           0.999000
Name: score_experimentally_determined_interaction, dtype: float64
count    116176.000000
mean          0.057293
std           0.121129
min           0.000000
25%           0.000000
50%           0.000000
75%           0.074000
max           0.999000
Name: experimentally_determined_interaction, dtype: float64
>= 0.041: 116176 pairs
>= 0.1: 18790 pairs
>= 0.15: 11408 pairs
>= 0.2: 8074 pairs
>= 0.3: 4174 pairs
>= 0.4: 2566 pairs
>= 0.5: 1866 pairs
>= 0.7: 1170 pairs
>= 0.9: 544 pairs
exactly 0: 70166 / 116176


In [7]:
# --- Experimentally-determined-interaction-only score ---
def single_channel_score(df, channel, prior=0.041):
    s = df[channel].values
    s_nop = np.where(s > 0, (s - prior) / (1 - prior), 0.0)
    s_nop = np.clip(s_nop, 0, None)
    return s_nop + prior * (1 - s_nop)

string['score_experimentally_determined_interaction'] = single_channel_score(
    string, 'experimentally_determined_interaction', p
)

exp_lenient_thresh = 0.1
exp_strict_thresh = 0.4

print('lenient (>= 0.1):', (string['score_experimentally_determined_interaction'] >= exp_lenient_thresh).sum())
print('strict  (>= 0.4):', (string['score_experimentally_determined_interaction'] >= exp_strict_thresh).sum())

lenient (>= 0.1): 18790
strict  (>= 0.4): 2566


## collapse to unique undirected pairs

In [9]:
string['pair_key'] = string.apply(lambda r: frozenset((r.node1, r.node2)), axis=1)
string_dedup = string.sort_values('combined_score_no_coexp', ascending=False).drop_duplicates(subset='pair_key')
print(f'{len(string)} rows -> {len(string_dedup)} unique unordered pairs')

116176 rows -> 58088 unique unordered pairs


In [10]:
'score_experimentally_determined_interaction' in string_dedup.columns

True

In [11]:
EXP_THRESHOLDS = {'lenient': 0.1, 'strict': 0.4}

exp_records = []
for _, row in string_dedup.iterrows():
    a, b = row.node1, row.node2
    for label, th in EXP_THRESHOLDS.items():
        if row.score_experimentally_determined_interaction >= th:
            exp_records.append({
                'partner1_ensembl_gene_id': a, 'partner2_ensembl_gene_id': b,
                'stringency': label,
            })

string_exp_final = pd.DataFrame(exp_records)
string_exp_final['partner1_symbol'] = string_exp_final.partner1_ensembl_gene_id.map(ens2sym)
string_exp_final['partner2_symbol'] = string_exp_final.partner2_ensembl_gene_id.map(ens2sym)
string_exp_final['partner1_role'] = 'Arbitrary'
string_exp_final['partner2_role'] = 'Arbitrary'
string_exp_final['directed'] = False
string_exp_final['stringency_detail'] = string_exp_final.stringency.map({
    'lenient': f'score_experimentally_determined_interaction >= {EXP_THRESHOLDS["lenient"]}',
    'strict':  f'score_experimentally_determined_interaction >= {EXP_THRESHOLDS["strict"]}',
})
string_exp_final['resource_name'] = 'STRING_experimental'

string_exp_final = string_exp_final[['partner1_symbol','partner1_ensembl_gene_id','partner2_symbol','partner2_ensembl_gene_id',
                                       'partner1_role','partner2_role','directed','stringency','stringency_detail','resource_name']]

print(string_exp_final.shape)
print(string_exp_final.stringency.value_counts())
string_exp_final.head()

(10678, 10)
stringency
lenient    9395
strict     1283
Name: count, dtype: int64


,partner1_symbol,partner1_ensembl_gene_id,partner2_symbol,partner2_ensembl_gene_id,partner1_role,partner2_role,directed,stringency,stringency_detail,resource_name
0,NDUFV1,ENSG00000167792,NDUFS2,ENSG00000158864,Arbitrary,Arbitrary,False,lenient,score_experimentally_determined_interaction >=...,STRING_experimental
1,NDUFV1,ENSG00000167792,NDUFS2,ENSG00000158864,Arbitrary,Arbitrary,False,strict,score_experimentally_determined_interaction >=...,STRING_experimental
2,EEF1D,ENSG00000104529,EEF1G,ENSG00000254772,Arbitrary,Arbitrary,False,lenient,score_experimentally_determined_interaction >=...,STRING_experimental
3,EEF1D,ENSG00000104529,EEF1G,ENSG00000254772,Arbitrary,Arbitrary,False,strict,score_experimentally_determined_interaction >=...,STRING_experimental
4,CDKN2D,ENSG00000129355,CDK6,ENSG00000105810,Arbitrary,Arbitrary,False,lenient,score_experimentally_determined_interaction >=...,STRING_experimental


## SAVE

In [12]:
string_exp_final[(string_exp_final.partner1_symbol.isin(['CCR2','CCR7'])) &
                  (string_exp_final.partner2_symbol.isin(['CCR2','CCR7']))]

,partner1_symbol,partner1_ensembl_gene_id,partner2_symbol,partner2_ensembl_gene_id,partner1_role,partner2_role,directed,stringency,stringency_detail,resource_name


In [13]:
string_exp_final.to_csv('/nfs/team151/mt19/annotation_preprocessed/string_experimental.tsv', sep='\t', index=False)
print(f'Saved {len(string_exp_final)} rows to string_experimental.tsv')

Saved 10678 rows to string_experimental.tsv


In [14]:
string_exp_lenient = string_exp_final[string_exp_final.stringency == 'lenient'].drop(columns=[]).copy()
string_exp_strict  = string_exp_final[string_exp_final.stringency == 'strict'].copy()

string_exp_lenient.to_csv('/nfs/team151/mt19/annotation_preprocessed/string_experimental_lenient.tsv', sep='\t', index=False)
string_exp_strict.to_csv('/nfs/team151/mt19/annotation_preprocessed/string_experimental_strict.tsv', sep='\t', index=False)

print(f'Saved {len(string_exp_lenient)} lenient rows, {len(string_exp_strict)} strict rows')

Saved 9395 lenient rows, 1283 strict rows


# 4. collectri

In [9]:
collectri_path = '/nfs/users/nfs_m/mt19/cardinal_analysis/ht/datasets/Flanders/trans_eQTLS_ERs/TF_target_pairs/CollecTRI_regulons.csv'
collectri = pd.read_csv(collectri_path)
print(collectri.shape)
print(collectri.columns.tolist())

(43175, 6)
['source', 'target', 'weight', 'resources', 'references', 'sign_decision']


## Sub-resource counting

In [10]:
def count_subresources(resources_str):
    parts = [p for p in resources_str.split(';') if p != 'CollecTRI']
    return len(set(parts))

collectri['n_subresources'] = collectri['resources'].apply(count_subresources)
print(collectri['n_subresources'].describe())

count    43175.000000
mean         1.612137
std          1.006267
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max          9.000000
Name: n_subresources, dtype: float64


## Mapping symbols to ensembl gene ids

In [11]:
def map_collectri_to_common_schema(df, stringency_label, stringency_detail):
    df = df.copy()
    df['source_ensembl'] = df.source.map(sym2ens)
    df['target_ensembl'] = df.target.map(sym2ens)
    mapped = df[df.source_ensembl.notna() & df.target_ensembl.notna()].copy()
    print(f'  {stringency_label}: {len(mapped)} / {len(df)} rows mapped')

    exploded = mapped.explode('source_ensembl').explode('target_ensembl')
    exploded = exploded[['source', 'source_ensembl', 'target', 'target_ensembl']].drop_duplicates()

    out = pd.DataFrame({
        'partner1_symbol': exploded.source,
        'partner1_ensembl_gene_id': exploded.source_ensembl,
        'partner2_symbol': exploded.target,
        'partner2_ensembl_gene_id': exploded.target_ensembl,
        'partner1_role': 'TF',
        'partner2_role': 'TF_target',
        'directed': True,
        'stringency': stringency_label,
        'stringency_detail': stringency_detail,
        'resource_name': 'CollecTRI',
    })
    return out

In [12]:
collectri_filtered = collectri[collectri.n_subresources >= 2].reset_index(drop=True)
print(f'strict (>=2 sub-resources): {len(collectri_filtered)} / {len(collectri)} rows')

collectri_strict = map_collectri_to_common_schema(collectri_filtered, 'strict', '>=2 CollecTRI sub-resources')
collectri_lenient = map_collectri_to_common_schema(collectri, 'lenient', 'full CollecTRI, no sub-resource filter')

print(collectri_strict.shape, collectri_lenient.shape)
collectri_strict.head()

strict (>=2 sub-resources): 16132 / 43175 rows
  strict: 15448 / 16132 rows mapped
  lenient: 41623 / 43175 rows mapped
(15463, 10) (41683, 10)


,partner1_symbol,partner1_ensembl_gene_id,partner2_symbol,partner2_ensembl_gene_id,partner1_role,partner2_role,directed,stringency,stringency_detail,resource_name
0,MYC,ENSG00000136997,TERT,ENSG00000164362,TF,TF_target,True,strict,>=2 CollecTRI sub-resources,CollecTRI
1,SMAD3,ENSG00000166949,JUN,ENSG00000177606,TF,TF_target,True,strict,>=2 CollecTRI sub-resources,CollecTRI
2,SMAD4,ENSG00000141646,JUN,ENSG00000177606,TF,TF_target,True,strict,>=2 CollecTRI sub-resources,CollecTRI
3,RELA,ENSG00000173039,FAS,ENSG00000026103,TF,TF_target,True,strict,>=2 CollecTRI sub-resources,CollecTRI
4,WT1,ENSG00000184937,NR0B1,ENSG00000169297,TF,TF_target,True,strict,>=2 CollecTRI sub-resources,CollecTRI


## SAVE

In [13]:
collectri_strict.to_csv('/nfs/team151/mt19/annotation_preprocessed/collectri_strict.tsv', sep='\t', index=False)
collectri_lenient.to_csv('/nfs/team151/mt19/annotation_preprocessed/collectri_lenient.tsv', sep='\t', index=False)
print(f'Saved {len(collectri_strict)} strict rows, {len(collectri_lenient)} lenient rows')

Saved 15463 strict rows, 41683 lenient rows


# 5. LIANA Ligand-receptor

In [14]:
lr_raw = pd.read_csv('/nfs/users/nfs_m/mt19/cardinal_analysis/ht/datasets/Flanders/trans_eQTLS_ERs/omni_resource.csv', index_col=0)
print(lr_raw.shape)
print(lr_raw.resource.value_counts())

(35360, 5)
resource
consensus           4624
mouseconsensus      3989
cellinker           3700
celltalkdb          3392
lrdb                3228
italk               2566
connectomedb2020    2266
cellchatdb          1912
ramilowski2015      1889
embrace             1650
baccin2019          1648
cellphonedb         1223
cellcall            1123
icellnet             738
guide2pharma         665
hpmr                 595
kirouac2010          152
Name: count, dtype: int64


##  mouse/complex exclusion:

In [15]:
lr_human = lr_raw[lr_raw.resource != 'mouseconsensus'].copy()
print('after excluding mouseconsensus:', len(lr_human), '/', len(lr_raw))

is_complex = lambda s: '_' in s
lr_human_single = lr_human[~lr_human.source_genesymbol.apply(is_complex) & ~lr_human.target_genesymbol.apply(is_complex)].copy()
print('after excluding complexes:', len(lr_human_single), '/', len(lr_human))

after excluding mouseconsensus: 31371 / 35360
after excluding complexes: 27569 / 31371


In [16]:
def map_lr_to_common_schema(df, resource_filter, stringency_label, stringency_detail):
    sub = df[df.resource == resource_filter].copy()
    sub['source_ensembl'] = sub.source_genesymbol.map(sym2ens)
    sub['target_ensembl'] = sub.target_genesymbol.map(sym2ens)
    mapped = sub[sub.source_ensembl.notna() & sub.target_ensembl.notna()].copy()
    print(f'  {stringency_label} ({resource_filter}): {len(mapped)} / {len(sub)} rows mapped')

    exploded = mapped.explode('source_ensembl').explode('target_ensembl')
    exploded = exploded[['source_genesymbol','source_ensembl','target_genesymbol','target_ensembl']].drop_duplicates()

    out = pd.DataFrame({
        'partner1_symbol': exploded.source_genesymbol,
        'partner1_ensembl_gene_id': exploded.source_ensembl,
        'partner2_symbol': exploded.target_genesymbol,
        'partner2_ensembl_gene_id': exploded.target_ensembl,
        'partner1_role': 'Ligand',
        'partner2_role': 'Receptor',
        'directed': True,
        'stringency': stringency_label,
        'stringency_detail': stringency_detail,
        'resource_name': 'Liana',
    })
    return out

liana_strict = map_lr_to_common_schema(lr_human_single, 'cellphonedb', 'strict', 'CellPhoneDB resource only')
liana_lenient = map_lr_to_common_schema(lr_human_single, 'consensus', 'lenient', 'Liana consensus resource')

print(liana_strict.shape, liana_lenient.shape)
liana_strict.head()

  strict (cellphonedb): 776 / 788 rows mapped
  lenient (consensus): 3503 / 3527 rows mapped
(777, 10) (3513, 10)


,partner1_symbol,partner1_ensembl_gene_id,partner2_symbol,partner2_ensembl_gene_id,partner1_role,partner2_role,directed,stringency,stringency_detail,resource_name
13086,JAG2,ENSG00000184916,NOTCH1,ENSG00000148400,Ligand,Receptor,True,strict,CellPhoneDB resource only,Liana
13087,DLL1,ENSG00000198719,NOTCH1,ENSG00000148400,Ligand,Receptor,True,strict,CellPhoneDB resource only,Liana
13088,IGF1,ENSG00000017427,IGF1R,ENSG00000140443,Ligand,Receptor,True,strict,CellPhoneDB resource only,Liana
13089,JAG1,ENSG00000101384,NOTCH1,ENSG00000148400,Ligand,Receptor,True,strict,CellPhoneDB resource only,Liana
13090,WNT5A,ENSG00000114251,FZD2,ENSG00000180340,Ligand,Receptor,True,strict,CellPhoneDB resource only,Liana


## SAVE

In [17]:
liana_strict.to_csv('/nfs/team151/mt19/annotation_preprocessed/liana_strict.tsv', sep='\t', index=False)
liana_lenient.to_csv('/nfs/team151/mt19/annotation_preprocessed/liana_lenient.tsv', sep='\t', index=False)
print(f'Saved {len(liana_strict)} strict rows, {len(liana_lenient)} lenient rows')

Saved 777 strict rows, 3513 lenient rows


# 6. POSTAR3 RBP-target transcripts

In [18]:
files = sorted(glob.glob('/nfs/team151/mt19/POSTAR3/per_chrom/*.csv'))
print(f'{len(files)} files found')

strict_pairs_list = []
lenient_pairs_list = []
for f in files:
    df = pd.read_csv(f)
    if len(df) == 0:
        print(f'{f.split("/")[-1]}: 0 rows, skipping')
        continue

    df_strict = df[df.passes_method_filter]
    strict_pairs = df_strict[['RBP_name', 'gene_id', 'gene_name']].drop_duplicates()
    strict_pairs_list.append(strict_pairs)

    lenient_pairs = df[['RBP_name', 'gene_id', 'gene_name']].drop_duplicates()
    lenient_pairs_list.append(lenient_pairs)

    print(f'{f.split("/")[-1]}: {len(df)} total, {len(strict_pairs)} strict unique pairs, {len(lenient_pairs)} lenient unique pairs')

postar3_strict_raw = pd.concat(strict_pairs_list, ignore_index=True).drop_duplicates()
postar3_lenient_raw = pd.concat(lenient_pairs_list, ignore_index=True).drop_duplicates()

print()
print('strict (passes_method_filter):', postar3_strict_raw.shape)
print('lenient (full):', postar3_lenient_raw.shape)

25 files found
chr10_rbp_gene_transcript_pairs.csv: 296921 total, 14136 strict unique pairs, 49322 lenient unique pairs
chr11_rbp_gene_transcript_pairs.csv: 481376 total, 17147 strict unique pairs, 68950 lenient unique pairs
chr12_rbp_gene_transcript_pairs.csv: 448799 total, 20085 strict unique pairs, 70636 lenient unique pairs
chr13_rbp_gene_transcript_pairs.csv: 129785 total, 7132 strict unique pairs, 23868 lenient unique pairs
chr14_rbp_gene_transcript_pairs.csv: 311268 total, 12327 strict unique pairs, 43742 lenient unique pairs
chr15_rbp_gene_transcript_pairs.csv: 272311 total, 11876 strict unique pairs, 43421 lenient unique pairs
chr16_rbp_gene_transcript_pairs.csv: 382088 total, 14924 strict unique pairs, 60771 lenient unique pairs
chr17_rbp_gene_transcript_pairs.csv: 552581 total, 21890 strict unique pairs, 80485 lenient unique pairs
chr18_rbp_gene_transcript_pairs.csv: 133684 total, 5883 strict unique pairs, 20893 lenient unique pairs
chr19_rbp_gene_transcript_pairs.csv: 48921

## Gene mapping

In [19]:
excluded_symbols = ['ACI', 'SDOS', 'SMN', 'HNRNPH']
rename_map = {
    'TDP43': 'TARDBP', 'U2AF65': 'U2AF2', 'UNR': 'CSDE1',
}

def map_postar3_to_common_schema(df, stringency_label, stringency_detail):
    df = df.rename(columns={'gene_id': 'substrate_gene_id', 'gene_name': 'substrate_gene_name'}).copy()
    df = df[~df.RBP_name.isin(excluded_symbols)].copy()
    df['RBP_name_mapped'] = df.RBP_name.replace(rename_map)
    df['RBP_ensembl_gene'] = df.RBP_name_mapped.map(sym2ens)

    n_unmapped = df.RBP_ensembl_gene.isna().sum()
    print(f'  {stringency_label}: {n_unmapped} / {len(df)} rows unmapped after exclusions/renames')
    df = df[df.RBP_ensembl_gene.notna()].copy()

    exploded = df.explode('RBP_ensembl_gene')
    exploded = exploded[['RBP_name', 'RBP_ensembl_gene', 'substrate_gene_id', 'substrate_gene_name']].drop_duplicates()

    out = pd.DataFrame({
        'partner1_symbol': exploded.RBP_name,
        'partner1_ensembl_gene_id': exploded.RBP_ensembl_gene,
        'partner2_symbol': exploded.substrate_gene_name,
        'partner2_ensembl_gene_id': exploded.substrate_gene_id,
        'partner1_role': 'RBP',
        'partner2_role': 'RBP_target',
        'directed': True,
        'stringency': stringency_label,
        'stringency_detail': stringency_detail,
        'resource_name': 'POSTAR3',
    })
    return out

postar3_strict = map_postar3_to_common_schema(postar3_strict_raw, 'strict', 'passes_method_filter == True (>=2 methods, >=1 confident: CLIPper/PureCLIP/PARalyzer)')
postar3_lenient = map_postar3_to_common_schema(postar3_lenient_raw, 'lenient', 'full POSTAR3, no method-count filter')

print(postar3_strict.shape, postar3_lenient.shape)
postar3_strict.head()

  strict: 0 / 335532 rows unmapped after exclusions/renames
  lenient: 18118 / 1272904 rows unmapped after exclusions/renames
(335532, 10) (1258775, 10)


,partner1_symbol,partner1_ensembl_gene_id,partner2_symbol,partner2_ensembl_gene_id,partner1_role,partner2_role,directed,stringency,stringency_detail,resource_name
89,AGO2,ENSG00000123908,COX15,ENSG00000014919,RBP,RBP_target,True,strict,"passes_method_filter == True (>=2 methods, >=1...",POSTAR3
90,AGO2,ENSG00000123908,ZMYND11,ENSG00000015171,RBP,RBP_target,True,strict,"passes_method_filter == True (>=2 methods, >=1...",POSTAR3
91,AGO2,ENSG00000123908,ZRANB1,ENSG00000019995,RBP,RBP_target,True,strict,"passes_method_filter == True (>=2 methods, >=1...",POSTAR3
92,AGO2,ENSG00000123908,ZDHHC6,ENSG00000023041,RBP,RBP_target,True,strict,"passes_method_filter == True (>=2 methods, >=1...",POSTAR3
93,AGO2,ENSG00000123908,ABCC2,ENSG00000023839,RBP,RBP_target,True,strict,"passes_method_filter == True (>=2 methods, >=1...",POSTAR3


In [20]:
unmapped_lenient = postar3_lenient_raw[~postar3_lenient_raw.RBP_name.isin(excluded_symbols)].copy()
unmapped_lenient['RBP_name_mapped'] = unmapped_lenient.RBP_name.replace(rename_map)
unmapped_lenient['mapped'] = unmapped_lenient.RBP_name_mapped.map(sym2ens).notna()

still_unmapped_symbols = unmapped_lenient[~unmapped_lenient.mapped].RBP_name.unique()
print(len(still_unmapped_symbols), 'distinct unmapped RBP symbols in lenient set')
print(sorted(still_unmapped_symbols))

3 distinct unmapped RBP symbols in lenient set
['AARS', 'DDX3', 'TROVE2']


In [21]:
candidates = {'AARS': 'AARS1', 'DDX3': 'DDX3X'}
for old, new in candidates.items():
    print(old, '->', new, ':', new in sym2ens)

print('TROVE2 in sym2ens directly:', 'TROVE2' in sym2ens)

AARS -> AARS1 : True
DDX3 -> DDX3X : True
TROVE2 in sym2ens directly: False


In [22]:
rename_map = {
    'TDP43': 'TARDBP', 'U2AF65': 'U2AF2', 'UNR': 'CSDE1',
    'AARS': 'AARS1', 'DDX3': 'DDX3X',
}

postar3_strict = map_postar3_to_common_schema(postar3_strict_raw, 'strict', 'passes_method_filter == True (>=2 methods, >=1 confident: CLIPper/PureCLIP/PARalyzer)')
postar3_lenient = map_postar3_to_common_schema(postar3_lenient_raw, 'lenient', 'full POSTAR3, no method-count filter')

print(postar3_strict.shape, postar3_lenient.shape)

  strict: 0 / 335532 rows unmapped after exclusions/renames
  lenient: 1988 / 1272904 rows unmapped after exclusions/renames
(335532, 10) (1274905, 10)


## SAVE

In [23]:
postar3_strict.to_csv('/nfs/team151/mt19/annotation_preprocessed/postar3_strict.tsv', sep='\t', index=False)
postar3_lenient.to_csv('/nfs/team151/mt19/annotation_preprocessed/postar3_lenient.tsv', sep='\t', index=False)
print(f'Saved {len(postar3_strict)} strict rows, {len(postar3_lenient)} lenient rows')

Saved 335532 strict rows, 1274905 lenient rows


# 7. Shedases and target substrates

In [24]:
merops = pd.read_csv('/nfs/team151/mt19/merops_db/merops_peptidase_substrate_ensembl_human.csv')
print(merops.shape)
print(merops.columns.tolist())
print(merops.cleavage_type.value_counts())

(14286, 8)
['code', 'peptidase_gene', 'substrate_uniprot', 'cleavage_evidence', 'cleavage_type', 'peptidase_ensembl_gene', 'peptidase_gene_mapped', 'substrate_ensembl_gene']
cleavage_type
non-physiological    8044
physiological        5860
synthetic             382
Name: count, dtype: int64


## Gene mapping

In [25]:
def map_merops_to_common_schema(df, stringency_label, stringency_detail):
    out = pd.DataFrame({
        'partner1_symbol': df.peptidase_gene,
        'partner1_ensembl_gene_id': df.peptidase_ensembl_gene,
        'partner2_symbol': df.substrate_ensembl_gene.map(ens2sym),
        'partner2_ensembl_gene_id': df.substrate_ensembl_gene,
        'partner1_role': 'Peptidase',
        'partner2_role': 'Substrate_peptidase',
        'directed': True,
        'stringency': stringency_label,
        'stringency_detail': stringency_detail,
        'resource_name': 'MEROPS',
    }).drop_duplicates()
    return out

merops_phys = merops[merops.cleavage_type == 'physiological']
merops_strict = map_merops_to_common_schema(merops_phys, 'strict', 'cleavage_type == physiological')
merops_lenient = map_merops_to_common_schema(merops, 'lenient', 'full MEROPS, all cleavage types (physiological/non-physiological/synthetic)')

print(merops_strict.shape, merops_lenient.shape)
merops_strict.head()

(5860, 10) (13915, 10)


,partner1_symbol,partner1_ensembl_gene_id,partner2_symbol,partner2_ensembl_gene_id,partner1_role,partner2_role,directed,stringency,stringency_detail,resource_name
1,CASP14,ENSG00000105141,EIF3J,ENSG00000104131,Peptidase,Substrate_peptidase,True,strict,cleavage_type == physiological,MEROPS
2,CASP14,ENSG00000105141,HSP90AB1,ENSG00000096384,Peptidase,Substrate_peptidase,True,strict,cleavage_type == physiological,MEROPS
3,CASP14,ENSG00000105141,TOP1,ENSG00000198900,Peptidase,Substrate_peptidase,True,strict,cleavage_type == physiological,MEROPS
4,CASP14,ENSG00000105141,HSP90B1,ENSG00000166598,Peptidase,Substrate_peptidase,True,strict,cleavage_type == physiological,MEROPS
5,CASP14,ENSG00000105141,SEM1,ENSG00000127922,Peptidase,Substrate_peptidase,True,strict,cleavage_type == physiological,MEROPS


## SAVE

In [26]:
merops_strict.to_csv('/nfs/team151/mt19/annotation_preprocessed/merops_strict.tsv', sep='\t', index=False)
merops_lenient.to_csv('/nfs/team151/mt19/annotation_preprocessed/merops_lenient.tsv', sep='\t', index=False)
print(f'Saved {len(merops_strict)} strict rows, {len(merops_lenient)} lenient rows')

Saved 5860 strict rows, 13915 lenient rows


# 8 Final check

In [27]:
import glob
final_files = sorted(glob.glob('/nfs/team151/mt19/annotation_preprocessed/*.tsv'))
for f in final_files:
    df = pd.read_csv(f, sep='\t')
    print(f.split('/')[-1], df.shape, df.resource_name.unique(), df.stringency.unique())

collectri_lenient.tsv (41683, 10) <ArrowStringArray>
['CollecTRI']
Length: 1, dtype: str <ArrowStringArray>
['lenient']
Length: 1, dtype: str
collectri_strict.tsv (15463, 10) <ArrowStringArray>
['CollecTRI']
Length: 1, dtype: str <ArrowStringArray>
['strict']
Length: 1, dtype: str
liana_lenient.tsv (3513, 10) <ArrowStringArray>
['Liana']
Length: 1, dtype: str <ArrowStringArray>
['lenient']
Length: 1, dtype: str
liana_strict.tsv (777, 10) <ArrowStringArray>
['Liana']
Length: 1, dtype: str <ArrowStringArray>
['strict']
Length: 1, dtype: str
merops_lenient.tsv (13915, 10) <ArrowStringArray>
['MEROPS']
Length: 1, dtype: str <ArrowStringArray>
['lenient']
Length: 1, dtype: str
merops_strict.tsv (5860, 10) <ArrowStringArray>
['MEROPS']
Length: 1, dtype: str <ArrowStringArray>
['strict']
Length: 1, dtype: str
postar3_lenient.tsv (1274905, 10) <ArrowStringArray>
['POSTAR3']
Length: 1, dtype: str <ArrowStringArray>
['lenient']
Length: 1, dtype: str
postar3_strict.tsv (335532, 10) <ArrowStringAr